# PbI₂ Speciation in a Helium Atmosphere

**Authors: Gianluca Nesti; Olha Marinich**

**Modified: G. Dan Miron**

Formatted with Claude Code

This example computes equilibrium speciation in a Pb–I–He system as a function of temperature (25–990 °C).

The model is inspired by:
> Liu et al. (2025), *Chemical speciation of iodine evaporated from liquid lead-bismuth eutectic investigated by thermosublimatography.  Journal of Radioanalytical and Nuclear Chemistry. Volume 334, pages 5201–5216 (2025)*.

**Objective:** Investigate the redistribution of Pb and I among condensed and gaseous species as temperature increases under a fixed helium atmosphere.

This type of calculation is relevant for:
- Nuclear reactor cover-gas chemistry
- Fission-product transport in liquid-metal systems
- High-temperature volatilisation processes

**What this notebook demonstrates:**
- Loading a thermodynamic project in xGEMS
- Defining elemental inventories
- Running equilibrium calculations over a temperature grid
- Extracting species amounts
- Post-processing with pandas and visualisation with matplotlib

**Chemical system exported from GEM-Selektor project found in gems-project**
- PbI2 G M1-6 3 0 1 300 0 single calculation
- Menu Data->Export GEMS3K files...-> Setup for exporting look-up arrays T, C from 0 to 1000 step 10

## Import python packages

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgems as xg

## Load the thermodynamic model

The GEMS3K files exported from GEM-Selektor are stored in the `gems_files/` folder.
The `.lst` file is the entry point that references the other JSON files.

In [ ]:
xg_system_filename = 'gems_files/M1-6-dat.lst'

xgEngine = xg.ChemicalEngine(xg_system_filename)

# preliminary check of thermodynamic consistency
code = xgEngine.reequilibrate()
print('GEMS return code:', code)

## Define element and species indexes

xGEMS works with index-based data tables. We retrieve the indexes once and reuse them throughout the calculation.

In [ ]:
# element indexes
element_indexes = {
    'Pb': xgEngine.indexElement('Pb'),
    'I':  xgEngine.indexElement('I'),
    'O':  xgEngine.indexElement('O'),
    'He': xgEngine.indexElement('He'),
}

# species indexes
species_indexes = {
    'Pb(s)':   xgEngine.indexSpecies('Pb'),
    'I2(s)':   xgEngine.indexSpecies('I2'),
    'I(g)':    xgEngine.indexSpecies('I(g)'),
    'I2(g)':   xgEngine.indexSpecies('I2(g)'),
    'Pb(g)':   xgEngine.indexSpecies('Pb(g)'),
    'Pb2(g)':  xgEngine.indexSpecies('Pb2(g)'),
    'PbI(g)':  xgEngine.indexSpecies('PbI(g)'),
    'PbI2(g)': xgEngine.indexSpecies('PbI2(g)'),
    'PbI2(s)': xgEngine.indexSpecies('PbI2(s)'),
}

print('Element indexes:', element_indexes)
print('Species indexes:', species_indexes)

## Define the system composition

We start from the exported bulk composition and overwrite the relevant elements.

| Element | Amount (mol) | Notes |
|---------|-------------|-------|
| Pb      | 1.61 × 10⁻⁵ | trace amount |
| I       | 2 × n(Pb)   | stoichiometric PbI₂ |
| He      | 0.75        | carrier gas |
| O       | 2 × 10⁻¹⁰  | trace oxygen |

Pressure is fixed at 1 × 10⁵ Pa (1 bar).

In [ ]:
b0 = xgEngine.elementAmounts().copy()

b0[element_indexes['Pb']] = 1.61e-5
b0[element_indexes['I']]  = 2 * b0[element_indexes['Pb']]
b0[element_indexes['He']] = 0.75
b0[element_indexes['O']]  = 2e-10

P = 1e5  # Pa

He_amount = b0[element_indexes['He']]
print(f'He amount: {He_amount} mol, P: {P} Pa')

### Alternative: define the composition with the `Material` class

`Material` lets you express a recipe in human-readable chemical formulas and units instead of working with raw element indexes. A `Material` can be passed directly to `equilibrate()` — no manual index bookkeeping needed.

Two `.add()` styles are available:
- `material.add("formula", amount, "unit")` — by chemical formula (e.g. `"PbI2"`, `"mol"` or `"g"`)
- `material.add({"Element": amount, ...})` — directly as element moles (dict, no unit = mol)

Materials can be combined with `+` to build a composite recipe. The result is identical to the index-based `b0` constructed above.

In [ ]:
from xgems import Material

# --- PbI2 solid source ---
pbi2_source = Material(xgEngine, "PbI2_source")
pbi2_source.add("PbI2", 1.61e-5, "mol")   # stoichiometric Pb + 2 I

# --- He carrier gas and trace O as element amounts (mol) ---
atmosphere = Material(xgEngine, "atmosphere")
atmosphere.add({"He": 0.75, "O": 2e-10})

# combine into one recipe
recipe = pbi2_source + atmosphere

# verify: compare with the index-based b0
b0_material = recipe.b()
print("b0 from indexes: ", b0)
print("b0 from Material:", b0_material)
print("max difference:  ", abs(b0 - b0_material).max())

# either b0 or recipe can be passed to equilibrate()
# xgEngine.equilibrate(T + 273.15, P, recipe)

## Temperature loop: equilibrium calculations

We sweep temperature from 25 °C to 990 °C in 10 °C steps and record the amount (mol) of each species of interest at each step.

xGEMS return codes:
- **2** — converged with automatic initial approximation (AIA)
- **6** — converged with smart initial approximation (SIA)

In [ ]:
temperatures = np.arange(25, 1000, 10)  # °C

results = {'T (°C)': [], 'status': []}
for name in species_indexes:
    results[name] = []

for T in temperatures:
    code = xgEngine.equilibrate(T + 273.15, P, recipe)

    results['T (°C)'].append(T)
    results['status'].append(code)

    for name, idx in species_indexes.items():
        results[name].append(xgEngine.speciesAmount(idx))

    if code not in (2, 6):
        print(f'Warning: equilibrium not fully converged at {T:.1f} °C (code {code})')

print('Calculation finished.')

## Collect results in a DataFrame

In [ ]:
df = pd.DataFrame(results)
df.to_csv('PbI2_speciation_results.csv', index=False)
df.head(10)

## Plot speciation vs. temperature

Expected trends:
- **PbI₂(s)** dominates at low temperature
- Increasing temperature promotes volatilisation → **PbI₂(g)**, **PbI(g)**, **I(g)**
- At very high temperatures, atomic gas-phase species become significant

In [ ]:
plt.rcParams.update({'font.size': 14})

fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(df['T (°C)'], df['PbI2(s)'], label='PbI₂(s)')
ax.plot(df['T (°C)'], df['PbI2(g)'], label='PbI₂(g)')
ax.plot(df['T (°C)'], df['PbI(g)'],  label='PbI(g)')
ax.plot(df['T (°C)'], df['I(g)'],    label='I(g)')
ax.plot(df['T (°C)'], df['Pb(g)'],   label='Pb(g)')

ax.set_xlabel('Temperature (°C)')
ax.set_ylabel('Amount (mol)')
ax.set_title(f'PbI₂ Speciation (He = {He_amount} mol, P = 1 bar)')
ax.legend()
ax.grid(True)

plt.tight_layout()
plt.savefig(f'PbI2_speciation_{He_amount}mol_He.png', dpi=150)
plt.show()

## Plot all species (log scale)

A logarithmic y-axis reveals minor species that are otherwise invisible on the linear plot.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

for name in species_indexes:
    ax.plot(df['T (°C)'], df[name], label=name)

ax.set_yscale('log')
ax.set_xlabel('Temperature (°C)')
ax.set_ylabel('Amount (mol)')
ax.set_title(f'PbI₂ Speciation – all species (log scale)')
ax.legend(fontsize=11)
ax.grid(True, which='both', ls='--', alpha=0.5)

plt.tight_layout()
plt.show()